# Methods for Temperature rescaling

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime as dt
import os

In [ ]:
from etdataset.dem import get_dem_from_roi
from etdataset.era5 import ERA5Dataset, download
from etdataset.icos import get_stations_config
from etdataset.plot import plot_dataset, plot_dem
from etdataset.temperature import RescalTempMethod, TempVariable, add_temp
from etdataset.utils import work_area_from_coord_point

In [ ]:
import logging

from etdataset.logging import LoggerManager

LoggerManager.set_level(logging.DEBUG)

## Inputs

Give a date

In [ ]:
date = dt.datetime(2021, 4, 4, 12, 0)

In [ ]:
station = "FR-Tou"

In [ ]:
cfg = get_stations_config(station)
lat = cfg.lat
lon = cfg.lon

Define a ROI

In [ ]:
roi_bbox_utm, roi_crs_utm = work_area_from_coord_point(
    lat,
    lon,
    8000,
    8000,
)["utm"]

In [ ]:
output_dir = "out"
os.makedirs(output_dir, exist_ok=True)

## Get data

In [ ]:
download(date=date, dataset=ERA5Dataset.ERA5, path=output_dir)
download(date=date, dataset=ERA5Dataset.ERA5PRESSURE, path=output_dir)

## Get DEM

In [ ]:
mnt_path = os.path.join(os.environ["MNT_PATH"], "DEM_Copernicus_30m/")
data = get_dem_from_roi(
    roi_bbox=roi_bbox_utm,
    roi_crs=roi_crs_utm,
    base_dir=mnt_path,
    resolution=60,
)

In [ ]:
# Add attributes
data.attrs["vis_date"] = date.date()
data.attrs["vis_time"] = date.time()
data.attrs["tir_date"] = date.date()
data.attrs["tir_time"] = date.time()
data

In [ ]:
plot_dem(data)

### Basic method : Rescaling with constant lapse rate

In [ ]:
updated_data = add_temp(
    data=data,
    dataset=ERA5Dataset.ERA5,
    variables=["ta", "td"],
    method=RescalTempMethod.CONST_LR,
    interp_type="cubic_spline",
    path=output_dir,
)

In [ ]:
plot_dataset(updated_data)

### Method 1 : Constant lapse rate per months

In [ ]:
updated_data = add_temp(
    data=data,
    dataset=ERA5Dataset.ERA5,
    variables=[TempVariable.TD, TempVariable.TA],
    method=RescalTempMethod.MONTHLY_LR,
    path=output_dir,
)

In [ ]:
plot_dataset(updated_data)

### Method 2 : compute lapse rate using fixed pressure levels and altitude threshold

In [ ]:
updated_data = add_temp(
    data=data,
    dataset=ERA5Dataset.ERA5,
    variables=["ta", "td"],
    method=RescalTempMethod.LR_PROFILE_FIXED_LEVELS,
    path=output_dir,
)

In [ ]:
plot_dataset(updated_data)

### Method 3 : compute lapse rate using different pressure levels and altitude threshold

In [ ]:
updated_data = add_temp(
    data=data,
    dataset=ERA5Dataset.ERA5,
    variables=[TempVariable.TD, TempVariable.TA],
    method=RescalTempMethod.LR_PROFILE_ALT_DEP,
    path=output_dir,
)

In [ ]:
plot_dataset(updated_data)

### Method 4 : Vertical interpolation from ERA5 pressure levels

In [ ]:
updated_data = add_temp(
    data=data,
    dataset=ERA5Dataset.ERA5,
    variables=[TempVariable.TD, TempVariable.TA],
    method=RescalTempMethod.VERTICAL_INTERP,
    path=output_dir,
)

In [ ]:
plot_dataset(updated_data)

### Method 5 : Vertical profile interpolation (pressure levels + surface level)

In [ ]:
updated_data = add_temp(
    data=data,
    dataset=ERA5Dataset.ERA5,
    variables=[TempVariable.TD, TempVariable.TA],
    method=RescalTempMethod.INTERP_PROFILE_SURF,
    path=output_dir,
)

In [ ]:
plot_dataset(updated_data)

### Method 6 : Hybrid interpolation combining verticla interpolation of pressure levels and lapse rate correction

In [ ]:
updated_data = add_temp(
    data=data,
    dataset=ERA5Dataset.ERA5,
    variables=[TempVariable.TD, TempVariable.TA],
    method=RescalTempMethod.INTERP_LR_HYBRID,
    path=output_dir,
)

In [ ]:
plot_dataset(updated_data)